In [ ]:
import os
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import numpy as np
from transformers import AutoProcessor, AutoModel

In [ ]:
_pickscore_model = None
_pickscore_processor = None
_device = "cuda" if torch.cuda.is_available() else "cpu"

def load_pickscore():
    """Singleton loader for the model to avoid re-loading on every function call."""
    global _pickscore_model, _pickscore_processor, _device
    
    if _pickscore_model is None:
        print("Loading PickScore model (this happens once)...")
        processor_path = "laion/CLIP-ViT-H-14-laion2B-s32B-b79K"
        model_path = "yuvalkirstain/PickScore_v1"
        
        _pickscore_processor = AutoProcessor.from_pretrained(processor_path)
        _pickscore_model = AutoModel.from_pretrained(model_path).eval().to(_device)
        print("Model loaded.")
        
    return _pickscore_processor, _pickscore_model

In [ ]:
def get_pick_scores(prompt, image_tensors):
    """
    Calculates preference probabilities for a batch of tensors.
    """
    processor, model = load_pickscore()
    
    # 1. Convert PyTorch Tensors (C, H, W) to PIL Images
    pil_images = []
    for img in image_tensors:
        # Normalize to 0-1 if not already
        img = img.detach().cpu().float()
        img = (img - img.min()) / (img.max() - img.min())
        # Convert to (H, W, C)
        img_np = img.permute(1, 2, 0).numpy()
        # Convert to uint8 0-255
        img_np = (img_np * 255).astype(np.uint8)
        pil_images.append(Image.fromarray(img_np))

    # 2. Preprocess (Your provided logic)
    image_inputs = processor(
        images=pil_images,
        padding=True,
        truncation=True,
        max_length=77,
        return_tensors="pt",
    ).to(_device)
    
    text_inputs = processor(
        text=prompt,
        padding=True,
        truncation=True,
        max_length=77,
        return_tensors="pt",
    ).to(_device)

    # 3. Calculate Scores (Your provided logic)
    with torch.no_grad():
        image_embs = model.get_image_features(**image_inputs)
        image_embs = image_embs / torch.norm(image_embs, dim=-1, keepdim=True)
    
        text_embs = model.get_text_features(**text_inputs)
        text_embs = text_embs / torch.norm(text_embs, dim=-1, keepdim=True)
    
        scores = model.logit_scale.exp() * (text_embs @ image_embs.T)[0]
        
        # Softmax gives probability relative to THIS batch of 16
        probs = torch.softmax(scores, dim=-1)
        
    return probs.cpu().tolist()

In [ ]:
def print_grid_text(title, values, rows=4, cols=4):
    """Helper to print the nice text grid."""
    print("\n" + "="*40)
    print(f" {title} ")
    print("="*40)
    for r in range(rows):
        start = r * cols
        end = start + cols
        row_vals = values[start:end]
        if len(row_vals) > 0:
            # Format: 8 characters wide, 4 decimals
            print(" ".join([f"{val:>8.4f}" for val in row_vals]))
    print("="*40)

In [ ]:
def plot_with_scores(folder_path, target_iterations, prompt, maximize_fitness=True, N=64):
    path = Path(folder_path).expanduser().resolve()
    
    nrows = int(np.sqrt(N))
    ncols = int(np.ceil(N / nrows))
    
    if not path.exists():
        print(f"[ERROR] Path not found: {path}")
        return

    if isinstance(target_iterations, (int, str)):
        target_iterations = [target_iterations]
    
    all_fitnesses = []
    all_images = []
    
    print(f"Loading data from: {path}")

    # --- 1. Load Data ---
    for itr in target_iterations:
        fit_path = path / f"{itr}_fitnesses.pt"
        img_path = path / f"{itr}_images.pt"
        
        if fit_path.exists() and img_path.exists():
            raw_fit = torch.load(fit_path, map_location='cpu')
            raw_img = torch.load(img_path, map_location='cpu')
            
            # Convert Numpy to Tensor if needed
            if isinstance(raw_fit, np.ndarray): raw_fit = torch.from_numpy(raw_fit)
            if isinstance(raw_img, np.ndarray): raw_img = torch.from_numpy(raw_img)

            all_fitnesses.append(raw_fit)
            all_images.append(raw_img)
        else:
            print(f"Warning: Files missing for iteration {itr}")

    if not all_fitnesses: return

    total_fitnesses = torch.cat(all_fitnesses)
    total_images = torch.cat(all_images)

    # --- 2. Sort & Select Top 16 ---
    sorted_indices = torch.argsort(total_fitnesses, descending=maximize_fitness)
    num_to_show = min(N, len(total_fitnesses))
    top_indices = sorted_indices[:num_to_show]
    
    top_images = total_images[top_indices]
    top_fitnesses = total_fitnesses[top_indices]

    # --- 3. Calculate PickScores ---
    print(f"Calculating PickScores for top {num_to_show} images...")
    pick_probs = get_pick_scores(prompt, top_images)

    # --- 4. Print Grids ---
    # Convert fitness tensors to list for the printer helper
    fit_list = top_fitnesses.tolist()
    
    print_grid_text("FITNESS (Matches Images)", fit_list, rows=nrows, cols=ncols)
    print("TOTAL FITNESS:", sum(fit_list))
    print("Best Fitness Image:", max(fit_list))
    print_grid_text(f"PICKSCORE PROBS ('{prompt}')", pick_probs, rows=nrows, cols=ncols)
    print("TOTAL PICKSCORE PROB:", sum(pick_probs))
    print("Best Fitness Image PickScore Prob:", max(pick_probs))

    # --- 5. Plot Images ---
    fig, axes = plt.subplots(nrows, ncols, figsize=(11, 12))
    itr_label = str(target_iterations) if len(target_iterations) < 4 else "Multiple"
    plt.suptitle(f"Top {N} by Fitness (Itrs: {itr_label})\nPrompt: '{prompt}'", fontsize=14)

    for i, ax in enumerate(axes.flat):
        if i < num_to_show:
            img = top_images[i]
            fit_val = fit_list[i]
            pick_val = pick_probs[i]
            
            # (C, H, W) -> (H, W, C)
            if img.dim() == 3 and img.shape[0] in [1, 3, 4]: 
                img = img.permute(1, 2, 0)
            
            # Normalize
            img = img.detach().cpu().numpy()
            img = (img - img.min()) / (img.max() - img.min())

            if img.shape[-1] == 1:
                ax.imshow(img.squeeze(), cmap='gray')
            else:
                ax.imshow(img)
                
            # Title: Fit on top, PickScore probability on bottom
            ax.set_title(f"Fit: {fit_val:.4f}\nPick: {pick_val:.4f}", fontsize=8)
        
        ax.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
folder_path = '/workspace/src/output/sdxl-turbo/29/dns/multi-hidden-layer-multi-svd/2025-12-03_150755_29/repertoire' 
prompt = "A photo of a palm tree made of water."
N = 64
plot_with_scores(folder_path, target_iterations=["125"], prompt=prompt, N= N)

In [ ]:
folder_path = '/workspace/src/output/sdxl-turbo/29/dns/random-seeds/2025-12-03_172023_29/repertoire'
plot_with_scores(folder_path, target_iterations=["25"], prompt=prompt,  N  = N)

In [ ]:
prompt = "children on a couch"
folder_path = "/workspace/src/output/sdxl-turbo/18/dns/multi-hidden-layer-multi-svd/2025-12-03_174544_18/repertoire"
plot_with_scores(folder_path, target_iterations=["125"], prompt=prompt,  N  = N)

In [ ]:
folder_path = "/workspace/src/output/sdxl-turbo/18/dns/random-seeds/2025-12-03_191937_18/repertoire"
plot_with_scores(folder_path, target_iterations=["25"], prompt=prompt,  N  = N)


In [ ]:
prompt = "a yield sign"
folder_path = "/workspace/src/output/sdxl-turbo/24/dns/multi-hidden-layer-multi-svd/2025-12-03_201000_24/repertoire"
plot_with_scores(folder_path, target_iterations=["125"], prompt=prompt,  N  = N)

In [ ]:
folder_path = "/workspace/src/output/sdxl-turbo/24/dns/random-seeds/2025-12-03_213617_24/repertoire"
plot_with_scores(folder_path, target_iterations=["25"], prompt=prompt,  N  = N)